In [ ]:


import sqlite3

# -------------------------------------------------
# 1. 메모리 DB 생성 + 테이블/데이터 세팅
# -------------------------------------------------
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE customers (
  customer_id TEXT,
  name TEXT,
  country TEXT,
  signup_date TEXT,
  grade TEXT
);
INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');

CREATE TABLE orders (
  order_id TEXT,
  customer_id TEXT,
  order_date TEXT,
  status TEXT,
  amount DECIMAL
);
INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
""")
conn.commit()

# -------------------------------------------------
# 2. 조회 쿼리 5개
#    (project_name.dataset_name. 접두사는 로컬 실행이라 뺐음)
# -------------------------------------------------
queries = {
    "Q1. Gold 등급이면서 Korea 국가인 고객": """
        SELECT customer_id, name, country, grade
        FROM customers
        WHERE grade = 'Gold'
          AND country = 'Korea';
    """,
    "Q2. 2023년 4분기 Paid 또는 Shipped 주문": """
        SELECT order_id, customer_id, order_date, status, amount
        FROM orders
        WHERE order_date BETWEEN '2023-10-01' AND '2023-12-31'
          AND (status = 'Paid' OR status = 'Shipped');
    """,
    "Q3. 국가 정보 없는 고객": """
        SELECT customer_id, name, country, grade
        FROM customers
        WHERE country IS NULL;
    """,
    "Q4. Gold/Silver 등급 중 취소/반품 주문": """
        SELECT o.order_id, c.customer_id, c.name, c.grade, o.status, o.amount
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.grade IN ('Gold', 'Silver')
          AND o.status IN ('Cancelled', 'Returned');
    """,
    "Q5. 결제금액 확정 + 20만원 이상 고액 주문": """
        SELECT order_id, customer_id, order_date, status, amount
        FROM orders
        WHERE amount IS NOT NULL
          AND amount >= 200000;
    """,
}

# -------------------------------------------------
# 3. 실행 및 결과 출력
# -------------------------------------------------
for title, sql in queries.items():
    print("=" * 60)
    print(title)
    print("=" * 60)
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    print(cols)
    for r in rows:
        print(r)
    print(f"(총 {len(rows)}건)\n")

conn.close()